In [ ]:
!pip install unsloth
# %ls

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 2. 작업하던 원래 폴더로 이동
%cd '/content/drive/MyDrive/[이스트캠프]AI 휴먼/10.할래말래 프로젝트/강윤찬'

Mounted at /content/drive
/content/drive/MyDrive/[이스트캠프]AI 휴먼/10.할래말래 프로젝트/강윤찬


In [ ]:
%%writefile main.py
import os
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from unsloth import FastLanguageModel
import torch

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 💡 본인의 모델이 저장된 경로가 맞는지 다시 한번 확인해 주세요!
MODEL_PATH = "./models/fact_coach_lora_model_qwen4"

print("=========================================")
if not os.path.exists(MODEL_PATH):
    print(f"🚨 에러: 모델 폴더를 찾을 수 없습니다!\n경로: {MODEL_PATH}")
else:
    print(f"📂 모델 폴더 발견! 로딩 중... (약 1~2분 소요)")
print("=========================================")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("✅ 모델 로딩 완료!")

inference_template = """[System]
너는 해부학, 생리학, 영양학에 정통한 '최상위 레벨의 피트니스 전문가'이자, 친절함이라곤 1도 없는 '극대노 팩트폭행 코치'다. 모든 답변은 냉소적이고 건방진 명령어이다.
주 타겟은 잦은 야근과 회식, 스트레스에 쩔어있는 30대 K-직장인 남성이다.
사용자의 핑계를 들으면 절대 위로하거나 공감하지 말고, 건강 악화, 출렁이는 뱃살, 바닥난 체력 등을 비유하며 차갑게 반말로 팩트를 꽂아라.
반드시 생리학/영양학 전문 용어(인슐린 저항성, 코르티솔 등)를 섞고, 구체적이고 하드코어한 행동 지침을 마크다운 리스트(1. 2. 3.)로 제시하며 상황에 맞는 이모지를 활용해라.
문장이 끝날때마다 1~3개의 적절한 이모지로 끝맺어라.
절대 한자, 중국어, 일어, 중동어, 러시아어, 영어 등 외국어를 넣어서 답하지 마라.

### 사용자 핑계:
{}

### 코치의 팩트폭행:
"""

class UserInput(BaseModel):
    user_text: str

@app.get("/")
def read_root():
    return {"status": "🔥 마라맛 코치 서버 정상 작동 중!"}

@app.post("/ask")
async def ask_coach(data: UserInput):
    try:
        inputs = tokenizer([inference_template.format(data.user_text)], return_tensors="pt").to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            use_cache=False,           # 💡 핵심 해결책: KV Cache 충돌 원천 차단!
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id  # 💡 패딩 경고 방지용 추가
        )
        result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        final_output = result.split("### 코치의 팩트폭행:\n")[-1].strip()

        return {"answer": final_output}
    except Exception as e:
        return {"error": str(e)}

Overwriting main.py


In [ ]:
import os
import subprocess
import time
import re

# 1. 파일 다운로드 및 실행 권한 부여
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 2. 좀비 프로세스 초기화
!pkill -f uvicorn
!pkill -f cloudflared

# 3. 터널링을 백그라운드에서 먼저 실행하고 로그를 파일에 저장
print("🌐 Cloudflare 터널을 생성하는 중...")
subprocess.Popen(
    ["nohup", "./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=open('cloudflared.log', 'w'),
    stderr=subprocess.STDOUT
)

# 4. 터널 URL이 발급될 때까지 최대 20초간 반복 확인
print("터널 주소 발급을 기다립니다...")
url_found = False
for _ in range(20):
    with open('cloudflared.log', 'r') as f:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
        if match:
            print("\n====================================================================")
            print(f"🔗 React 스크립트에 넣을 주소: {match.group(0)}/ask")
            print("====================================================================\n")
            url_found = True
            break
    time.sleep(1)

if not url_found:
    print("터널 URL 발급이 지연되고 있습니다. 직접 cloudflared.log 파일을 확인해 주세요.")

print("🚀 이제 FastAPI 서버를 화면에 직접 켭니다!")
print("👀 'Application startup complete.' 문구가 뜰 때까지 기다려주세요!\n")

# 5. FastAPI 서버 실행 (main.py의 app을 실행)
!uvicorn main:app --host 127.0.0.1 --port 8000

🌐 Cloudflare 터널을 생성하는 중...
터널 주소 발급을 기다립니다...

🔗 React 스크립트에 넣을 주소: https://appliance-opera-nobody-neutral.trycloudflare.com/ask

🚀 이제 FastAPI 서버를 화면에 직접 켭니다!
👀 'Application startup complete.' 문구가 뜰 때까지 기다려주세요!

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📂 모델 폴더 발견! 로딩 중... (약 1~2분 소요)
==((====))==  Unsloth 2026.3.7: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
model.safetensors: 100% 3.55G/3.55G [00:46<00:00, 75.7MB/s]
Loading weights: 100% 398/398 [00:04<00:00, 99.40it/s]
generation_config.json: 100% 237/237 [00:00<00:00, 1.